# Sleep Scoring Model
Dataset : sleep_logs.csv

Target : memprediksi seberapa baik kualitas tidur seseorang (dalam bentuk angka skor) berdasarkan 3 inputan yaitu durasi tidur, jumlah interupsi, dan hutang tidur yang mereka miliki dengan output {"quality_score": [nilai]}

Model : Feed-Forward Neural Network

# Persiapan & Import Library

In [ ]:
# %load_ext tensorboard
import os
import datetime
import json
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.20.0


In [ ]:
!pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


# Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/data-sleep-score/sleep_logs.csv'
df = pd.read_csv(path)

df.head()

,id,user_id,date,bedtime,wake_time,duration_hours,interruptions,sleep_debt_hours,quality_score
0,slp_26400,26400,2026-04-20,00:30,08:45,8.24,5,0.00,0.6
1,slp_111583,11583,2026-05-07,21:33,04:30,6.94,1,1.06,0.9
2,slp_87840,87840,2026-04-20,20:43,03:57,7.22,8,0.78,0.4
3,slp_82065,82065,2026-05-05,00:09,07:29,7.33,3,0.67,0.6
4,slp_35311,35311,2026-05-19,19:12,04:43,9.51,1,0.00,0.8


In [ ]:
# save di storage sementara di colab
df.to_csv('sleep_logs.csv', index=False)

# Preprocessing Data

In [ ]:
# Load dataset
df = pd.read_csv('sleep_logs.csv')

# Pisahkan Fitur (X) dan Target (y)
features = ['duration_hours', 'interruptions', 'sleep_debt_hours']
X = df[features].values
y = df['quality_score'].values.reshape(-1, 1)

# Split Data: Train 70%, Val 15%, Test 15%
# Pisahkan 70% Train dan 30% Sisa (Temporary)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

# Pisahkan 30% sisa menjadi Validation (15%) dan Test (15%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# Normalisasi Fitur menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Data Train      : {X_train_scaled.shape}")
print(f"Data Validation : {X_val_scaled.shape}")
print(f"Data Test       : {X_test_scaled.shape}")

Data Train      : (4550, 3)
Data Validation : (975, 3)
Data Test       : (975, 3)


In [ ]:
print(X_train_scaled.shape[1])

3


# Menyimpan Dataset Split

In [ ]:
# Saving split dataset

# Menggunakan data yang SUDAH di-scale agar mudah untuk training ulang
train_df = pd.DataFrame(X_train_scaled, columns=features)
train_df['quality_score'] = y_train

val_df = pd.DataFrame(X_val_scaled, columns=features)
val_df['quality_score'] = y_val

test_df = pd.DataFrame(X_test_scaled, columns=features)
test_df['quality_score'] = y_test

# Ekspor ke file CSV di lokal Colab
train_df.to_csv('X_train_clean.csv', index=False)
val_df.to_csv('X_val_clean.csv', index=False)
test_df.to_csv('X_test_clean.csv', index=False)

print("Dataset Train, Val, dan Test berhasil disimpan sebagai CSV!")

Dataset Train, Val, dan Test berhasil disimpan sebagai CSV!


In [ ]:
from google.colab import files

# Download dataset hasil split
files.download('X_train_clean.csv')
files.download('X_val_clean.csv')
files.download('X_test_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Membangun Arsitektur Feed-Forward NN

In [ ]:
# Definisikan Arsitektur Deeper Model
def build_deeper_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=(input_shape,)),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='linear')  # Prediksi continuous quality score
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model

model = build_deeper_model(input_shape=X_train_scaled.shape[1])
model.summary()

# Setup Log Direktori untuk TensorBoard
log_dir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback = callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_5 (Dense)                 │ (None, 256)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,289 (173.00 KB)

 Trainable params: 44,289 (173.00 KB)

 Non-trainable params: 0 (0.00 B)

# Konfigurasi TensorBoard & Callbacks

In [ ]:
from tensorflow.keras import callbacks

# memperkecil learning rate secara otomatis
lr_reducer = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,           # Jika 5 epoch tidak membaik, perkecil LR
    min_lr=0.00001,
    verbose=1
)

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=25,          # Naikkan menjadi 25 epoch agar model punya waktu mencoba dengan LR baru
    restore_best_weights=True
)

# Training Model

In [ ]:
EPOCHS = 200
BATCH_SIZE = 16

history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[tensorboard_callback, early_stopping],
    verbose=1
)

Epoch 1/200
285/285 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0406 - mae: 0.1519 - val_loss: 0.0250 - val_mae: 0.1249
Epoch 2/200
285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0260 - mae: 0.1257 - val_loss: 0.0264 - val_mae: 0.1241
Epoch 3/200
285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0261 - mae: 0.1256 - val_loss: 0.0241 - val_mae: 0.1201
Epoch 4/200
285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0257 - mae: 0.1247 - val_loss: 0.0241 - val_mae: 0.1207
Epoch 5/200
285/285 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0258 - mae: 0.1250 - val_loss: 0.0246 - val_mae: 0.1234
Epoch 6/200
285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0254 - mae: 0.1243 - val_loss: 0.0241 - val_mae: 0.1220
Epoch 7/200
285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0254 - mae: 0.1237 - val_loss: 0.0241 - val_mae: 0.1211
Epoch 8/200
285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0254 - mae: 0.1240 - val_loss: 0.0246 - val_mae: 0.1220
Epoch 9/200
285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/

In [ ]:
# mengarsipkan logs tensorboard ke lokal

import shutil
from google.colab import files

# Kompres folder 'logs' menjadi file 'tensorboard_logs.zip'
shutil.make_archive('tensorboard_logs', 'zip', 'logs')
print("✅ Folder logs berhasil dikompres menjadi tensorboard_logs.zip")

# download otomatis
files.download('tensorboard_logs.zip')

✅ Folder logs berhasil dikompres menjadi tensorboard_logs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Evaluasi Model

In [ ]:
test_loss, test_mae = model.evaluate(X_test_scaled, y_test, verbose=0)

print("=== HASIL EVALUASI DATA TEST ===")
print(f"Test Loss (MSE) : {test_loss:.5f}")
print(f"Test MAE        : {test_mae:.5f}\n")

if test_mae < 0.02:
    print("✅ Sukses! Model memenuhi target akurasi MAE < 0.02")
else:
    print("❌ Model belum memenuhi target. Coba sesuaikan arsitektur atau naikkan Epoch.")

# scaler
joblib.dump(scaler, 'sleep_scaler.pkl')
print("\n✅ Scaler berhasil diamankan ke penyimpanan lokal: sleep_scaler.pkl")

=== HASIL EVALUASI DATA TEST ===
Test Loss (MSE) : 0.02199
Test MAE        : 0.11620

❌ Model belum memenuhi target. Coba sesuaikan arsitektur atau naikkan Epoch.

✅ Scaler berhasil diamankan ke penyimpanan lokal: sleep_scaler.pkl

=== MEMULAI PROSES EKSPOR MODEL ===


In [ ]:
print("\n=== MEMULAI PROSES EKSPOR MODEL ===")

# export format .h5
model.save('sleep_scoring_model.h5')
print("✅ Model sukses diekspor ke format Keras Legacy: sleep_scoring_model.h5")


# export ke format tensorflow savedmodel(Folder)
try:
    model.export('sleep_scoring_savedmodel')
except AttributeError:
    model.save('sleep_scoring_savedmodel', save_format='tf')
print("✅ Model sukses diekspor ke format TensorFlow SavedModel: folder 'sleep_scoring_savedmodel/'")


# konversi dan export ke format .tflite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('sleep_scoring_model.tflite', 'wb') as f:
    f.write(tflite_model)
print("✅ Model sukses dikonversi dan diekspor ke format TF Lite: sleep_scoring_model.tflite")


=== MEMULAI PROSES EKSPOR MODEL ===
✅ Model sukses diekspor ke format Keras Legacy: sleep_scoring_model.h5
Saved artifact at 'sleep_scoring_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3), dtype=tf.float32, name='keras_tensor_6')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139564719341008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139564719341200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139564719330448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139564566359376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139564566349392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139564566360720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139564566361872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139564566355152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139564566356688: TensorSpec(shape=(), dtype=tf

# Script Inference (Output JSON Format)

In [ ]:
%%writefile inference_sleep.py
import json
import numpy as np
import tensorflow as tf
import joblib

def run_inference(duration_hours, interruptions, sleep_debt_hours):
    # Reload model dan scaler yang disimpan
    loaded_model = tf.keras.models.load_model('sleep_scoring_model.h5', compile=False)
    loaded_scaler = joblib.load('sleep_scaler.pkl')

    # Bungkus data mentah ke array 2 dimensi (3 fitur)
    raw_input = np.array([[duration_hours, interruptions, sleep_debt_hours]])

    # Lakukan transformasi scaling fitur
    scaled_input = loaded_scaler.transform(raw_input)

    # Prediksi nilai kualitas tidur
    prediction = loaded_model.predict(scaled_input, verbose=0)

    # Ambil nilai skalar dari numpy array hasil prediksi
    score_value = float(prediction[0][0])

    # Batasi skor pada rentang logis data [0.0 s.d 1.0]
    score_value = max(0.0, min(1.0, score_value))

    # Konversi hasil akhir menjadi format JSON string
    output_dict = {"quality_score": [round(score_value, 2)]}
    return json.dumps(output_dict)

Writing inference_sleep.py


In [ ]:
from inference_sleep import run_inference

if __name__ == "__main__":
    # Simulasi User: Tidur 8.24 jam, interupsi terbangun 5 kali, hutang tidur 0 jam

    json_result = run_inference(duration_hours=8.24, interruptions=5, sleep_debt_hours=0.0)
    print("Output JSON dari file inference_sleep.py:")
    print(json_result)

Output JSON dari file inference_sleep.py:
{"quality_score": [0.58]}


# Download semua file-file ke lokal

In [ ]:
import shutil
from google.colab import files

print("Mempersiapkan pengunduhan seluruh berkas proyek...")

# Kompres folder SavedModel dan Logs menjadi ZIP
shutil.make_archive('sleep_scoring_savedmodel', 'zip', 'sleep_scoring_savedmodel')
shutil.make_archive('tensorboard_logs', 'zip', 'logs')

# download semua berkas ke lokal komputer
files.download('sleep_scoring_model.h5')
files.download('sleep_scoring_savedmodel.zip')
files.download('sleep_scoring_model.tflite')
files.download('tensorboard_logs.zip')
files.download('sleep_scaler.pkl')
files.download('inference_sleep.py')  # <-- File script inference

print("📦 Sukses! Tunggu beberapa saat, browser Anda akan otomatis mendownload 6 file di atas.")

Mempersiapkan pengunduhan seluruh berkas proyek...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📦 Sukses! Tunggu beberapa saat, browser Anda akan otomatis mendownload 6 file di atas.
